# Resaving GPM data in table format

In [ ]:
import pandas as pd
import h5py
import os
import pandas as pd
import plotly.express as px

In [ ]:
input_directory = "/home/kainis/electro_cloud_distrib/data/GPM/CMB_2014-2018"

In [ ]:
cols_to_keep = ["KuGMI/ScanTime/Year",
    "KuGMI/ScanTime/Month",
    "KuGMI/ScanTime/DayOfMonth",
    "KuGMI/ScanTime/Hour",
    "KuGMI/ScanTime/Minute",
    "KuGMI/ScanTime/Second",
    "KuGMI/ScanTime/MilliSecond",
    "KuGMI/ScanTime/DayOfYear",
    "KuGMI/ScanTime/SecondOfDay",
    "KuGMI/Latitude",
    "KuGMI/Longitude",
    "KuGMI/tenMeterWindSpeed"]

In [ ]:
def extract_datasets(h5_obj, prefix=''):
    data = {}
    for key in h5_obj:
        item = h5_obj[key]
        path = f"{prefix}/{key}" if prefix else key
        if isinstance(item, h5py.Dataset) & (path in cols_to_keep):
            data[path] = item[()]
        elif isinstance(item, h5py.Group):
            data.update(extract_datasets(item, path))
    return data

In [ ]:
def dict_with_arrays_to_df(data_dict):
    processed = {k: list(v) for k, v in data_dict.items()}
    return pd.DataFrame(processed)

In [ ]:
def process_extacted_columns(df):
    df["timestamp_str"] = df.apply(lambda x: f'{x["KuGMI/ScanTime/Year"]}-{x["KuGMI/ScanTime/Month"]}-{x["KuGMI/ScanTime/DayOfMonth"]} {x["KuGMI/ScanTime/Hour"]}:{x["KuGMI/ScanTime/Minute"]}:{x["KuGMI/ScanTime/Second"]}', axis=1)
    df["timestamp"] = df["timestamp_str"].apply(pd.to_datetime)
    df.rename(columns={"KuGMI/Latitude": "latitude", "KuGMI/Longitude": "longitude", "KuGMI/tenMeterWindSpeed": "tenMeterWindSpeed"}, inplace=True)
    res_df = df[["timestamp", "latitude", "longitude", "tenMeterWindSpeed"]].explode(["latitude", "longitude", "tenMeterWindSpeed"])
    return res_df

In [ ]:
res_df = pd.DataFrame()
for root, dir, files in os.walk(input_directory):
    for file in files:
        if file.endswith("HDF5"):
            print("processing", file)
            with h5py.File(os.path.join(root, file), 'r') as hdf_file:
                extracted_columns = extract_datasets(hdf_file)
                raw_df = dict_with_arrays_to_df(extracted_columns)
                processed_df = process_extacted_columns(raw_df)
                res_df = pd.concat([res_df, processed_df])

In [ ]:
res_df.latitude = res_df.latitude.astype(float)
res_df.longitude = res_df.longitude.astype(float)
res_df.tenMeterWindSpeed = res_df.tenMeterWindSpeed.astype(float)
res_df.sort_values("timestamp", inplace=True)

In [ ]:
res_df.to_parquet("/home/kainis/electro_cloud_distrib/data/GPM/CMB_2014_2018.parquet")

In [ ]:
res_df_sample = res_df.sort_values("timestamp")[:10000]

In [ ]:
fig = px.scatter_geo(
    res_df_sample,
    lat='latitude',
    lon='longitude',
    color='tenMeterWindSpeed',
    animation_frame=res_df_sample['timestamp'].astype(str),  # animation frame must be string
    projection="natural earth",
    color_continuous_scale="Viridis",
    range_color=(res_df_sample['tenMeterWindSpeed'].min(), res_df_sample['tenMeterWindSpeed'].max()),
    title="Wind Speed Over Time by Location",
    size_max=10
)

fig.update_layout(geo=dict(showland=True))
fig.show()
fig.write_html("/home/kainis/electro_cloud_distrib/data/GPM/satellite_dznamic_coverage.html")

In [ ]:
fig = px.scatter_geo(
    res_df[res_df.timestamp < pd.to_datetime("2015-01-01 00:00:00")],
    lat='latitude',
    lon='longitude',
    color='tenMeterWindSpeed',
    hover_name='timestamp',
    projection="natural earth",
    color_continuous_scale="Viridis",
    title="Wind Speed Observations (2014)"
)

fig.update_layout(geo=dict(showland=True))
fig.show()

In [ ]:
res_df.timestamp.drop_duplicates().head(40)